# 02 — Veri Tanıtımı ve Tanımlayıcı İstatistik

Bu notebook hocanın istediği **veri seti tanıtımını** üretir: veri kaynağı, örnek ve özellik sayısı, sınıf dağılımı, her özellik için ortalama / medyan / mod / standart sapma / min / max / çeyrekler, grafikler. Ayrıca eksik değer, aykırı değer ve sınıf dengesizliği kontrollerini yapıp ön işleme kararlarını gerekçelendirir.

**Earth Engine kullanılmaz.** Yalnızca 01 notebook'unun ürettiği CSV dosyaları okunur.

**Tanımlayıcı istatistik nedir?** Veriyi modellemeden önce "bu veri neye benziyor?" sorusunu sayılarla cevaplamaktır. Ortalama ve medyan merkezi, standart sapma ve çeyrekler yayılımı gösterir. Ortalama ile medyan çok farklıysa dağılım çarpıktır ya da aykırı değerler vardır.

> Her grafiğin altında **"Neye bakmalıyız"** notu var. Sonuçlar notebook çalıştıktan sonra belli olacağı için yorumları sayılara bakarak siz yazacaksınız. Not, neyi kontrol edeceğinizi söylüyor.

## Kurulum: kod GitHub'dan, sonuçlar Drive'a
- **Kod** (`config.py`, `src/`) özel GitHub reposundan Colab'ın geçici diskine (`/content/yapay-zeka-proje`) çekilir. Her çalıştırmada en güncel sürüm alınır (`git pull`).
- **Veri ve sonuçlar** (`data/`, `results/`) Google Drive'da `Drive'ım/yapay-zeka-proje/` altına yazılır. Colab kapansa da kaybolmaz.
- Özel repoyu okumak için Colab'ın **Gizli anahtarlar (Secrets)** bölümündeki `GITHUB_TOKEN` kullanılır. Token ekrana yazdırılmaz.

In [ ]:
import os, sys, subprocess, importlib

GITHUB_REPO = "selinzeykaya/yapay-zeka-proje"
CODE_DIR = "/content/yapay-zeka-proje"                 # kod (GitHub'dan)
WORK_DIR = "/content/drive/MyDrive/yapay-zeka-proje"   # veri + sonuçlar (Drive)

def _git(args, token):
    r = subprocess.run(["git"] + args, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git hatası: " + (r.stderr or r.stdout).replace(token, "***"))

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception as e:
        raise RuntimeError("Colab'da sol menüdeki anahtar simgesinden GITHUB_TOKEN ekleyin ve "
                           "'Not defteri erişimi'ni açın (README'ye bakın).") from e
    if os.path.exists(os.path.join(CODE_DIR, ".git")):
        _git(["-C", CODE_DIR, "pull", "-q"], token)
    else:
        _git(["clone", "-q", f"https://{token}@github.com/{GITHUB_REPO}.git", CODE_DIR], token)
    os.makedirs(WORK_DIR, exist_ok=True)
else:   # yerel bilgisayarda: notebooks/ klasörünün bir üstü
    CODE_DIR = WORK_DIR = os.path.abspath("..")

sys.path.insert(0, CODE_DIR)
os.chdir(WORK_DIR)
import config
importlib.reload(config)
for d in [config.DATA_RAW, config.DATA_PROCESSED, config.TABLES_DIR, config.FIGURES_DIR, config.MODELS_DIR]:
    os.makedirs(d, exist_ok=True)
commit = subprocess.run(["git", "-C", CODE_DIR, "log", "-1", "--format=%h %cd", "--date=short"],
                        capture_output=True, text=True).stdout.strip()
print("Kod     :", CODE_DIR, "| sürüm:", commit)
print("Sonuçlar:", WORK_DIR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
T, F = config.TABLES_DIR, config.FIGURES_DIR

def savefig(name):
    plt.tight_layout()
    plt.savefig(f"{F}/{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

df = pd.read_csv(f"{config.DATA_PROCESSED}/samples_2021.csv")
FEATS = config.FEATURES
CLASS_ORDER = [config.CLASSES[k] for k in sorted(config.CLASSES)]
CITY_ORDER = list(config.CITIES)
print(df.shape)
df.head()

## 1. Veri setinin özeti

In [ ]:
summary = pd.DataFrame([
    ("Girdi verisi", "Landsat 8/9 Collection 2 Level-2, yaz (Haz–Ağu) medyan kompoziti, 30 m"),
    ("Etiket verisi", f"ESA WorldCover v200 ({config.LABEL_YEAR}), 6 sınıfa indirgenmiş"),
    ("Çalışma alanı", ", ".join(CITY_ORDER)),
    ("Örnek (nokta) sayısı", len(df)),
    ("Özellik sayısı (veri)", len(FEATS)),
    ("Özellik sayısı (model girdisi)", len(config.MODEL_FEATURES)),
    ("Model girdisi", ", ".join(config.MODEL_FEATURES) + "  (NDMI = −NDBI olduğu için çıkarıldı)"),
    ("Sınıf sayısı", df["class"].nunique()),
    ("Sınıflar", ", ".join(CLASS_ORDER)),
    ("Örnekleme", f"Sınıf dengeli, noktalar arası ≥ {config.MIN_POINT_DISTANCE_M} m, 3×3 homojen pikseller"),
], columns=["Özellik", "Değer"])
summary.to_csv(f"{T}/veri_seti_ozeti.csv", index=False, encoding="utf-8-sig")
summary

## 2. Sınıf dağılımı
Örnekler bilerek **sınıf dengeli** alındı, bu yüzden her şehirde sınıfların sayısı birbirine yakın olmalı. Bir şehirde bir sınıf hedefin (~333) altındaysa o ilde o sınıf doğada nadirdir.

Karşılaştırma için gerçek arazi dağılımını da veriyoruz: WorldCover'a göre her ilin yüzde kaçı hangi sınıf. Bu iki tablo arasındaki fark önemli. Model dengeli veriyle eğitiliyor ama gerçek haritada sınıflar çok dengesiz.

In [ ]:
dist = pd.crosstab(df["city"], df["class_name"]).reindex(index=CITY_ORDER, columns=CLASS_ORDER).fillna(0).astype(int)
dist["Toplam"] = dist.sum(axis=1)
dist.loc["Toplam"] = dist.sum()
dist.to_csv(f"{T}/sinif_dagilimi.csv", encoding="utf-8-sig")
display(dist)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
dist.drop(index="Toplam", columns="Toplam").plot(kind="bar", ax=axes[0], width=0.8, colormap="tab10")
axes[0].set_title("Örnek sayısı: şehir × sınıf"); axes[0].set_ylabel("Nokta sayısı"); axes[0].set_xlabel("")
axes[0].legend(title="Sınıf", fontsize=8); axes[0].tick_params(axis="x", rotation=30)
overall = dist.loc["Toplam", CLASS_ORDER]
axes[1].bar(CLASS_ORDER, overall, color=sns.color_palette("tab10", len(CLASS_ORDER)))
for i, v in enumerate(overall):
    axes[1].text(i, v, f"{v}\n(%{100 * v / overall.sum():.1f})", ha="center", va="bottom", fontsize=8)
axes[1].set_title("Toplam sınıf dağılımı"); axes[1].set_ylabel("Nokta sayısı"); axes[1].tick_params(axis="x", rotation=30)
savefig("sinif_dagilimi")

**Neye bakmalıyız:** Çubuklar şehir içinde aşağı yukarı eşit mi? Hangi şehir-sınıf çiftleri belirgin şekilde kısa? Kısa olanlar o ilde nadir bulunan arazi tipleridir (bkz. `ornek_sayilari.csv`). Bu sınıflarda modelin o şehirdeki performansı daha belirsiz olacaktır.

In [ ]:
wc_pct = pd.read_csv(f"{T}/worldcover_sinif_alanlari_yuzde.csv", index_col=0).reindex(CITY_ORDER)
wc_pct = wc_pct.loc[:, wc_pct.max() > 0.05]     # hiçbir ilde %0.05'i geçmeyen sınıfları gösterme
ax = wc_pct.plot(kind="barh", stacked=True, figsize=(12, 5), colormap="tab20", width=0.8)
ax.set_xlabel("İl alanının yüzdesi (%)"); ax.set_ylabel("")
ax.set_title("Gerçek arazi örtüsü dağılımı (ESA WorldCover 2021)")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
savefig("worldcover_gercek_dagilim")

**Neye bakmalıyız:** Her ilde baskın arazi tipi hangisi (ör. Konya ve Şanlıurfa'da tarım, Artvin'de orman bekleniyor)? Yapılaşma ve su en çok hangi illerde? Bu grafik çalışma alanı seçimini gerekçelendirir. Grafikteki iller birbirinden ne kadar farklıysa, bölgeler arası genelleme (Faz 6) o kadar zor olur.

## 3. Tanımlayıcı istatistikler
Her özellik için: ortalama, medyan, mod, standart sapma, min, max, çeyrekler (Q1 = %25, Q3 = %75), çarpıklık ve eksik değer sayısı.

**Mod hakkında not:** Bantlar ve indeksler sürekli sayılardır ve neredeyse hiçbir değer iki kez tam aynı tekrarlanmaz. Bu yüzden ham veride mod anlamsızdır. Modu, değerleri **2 ondalık basamağa yuvarlayarak** hesaplıyoruz: "en sık görülen değer aralığı" gibi düşünün.

**Çarpıklık (skewness):** 0'a yakınsa dağılım simetrik. Pozitifse sağ kuyruk uzun (birkaç çok büyük değer var), negatifse sol kuyruk uzun.

In [ ]:
def describe_full(d):
    out = pd.DataFrame({
        "ortalama": d.mean(), "medyan": d.median(),
        "mod (2 ondalık)": d.round(2).mode().iloc[0],
        "std": d.std(), "min": d.min(), "Q1": d.quantile(0.25),
        "Q3": d.quantile(0.75), "max": d.max(),
        "carpiklik": d.skew(), "eksik": d.isna().sum(),
    })
    return out

desc = describe_full(df[FEATS]).round(4)
desc.to_csv(f"{T}/tanimlayici_istatistik.csv", encoding="utf-8-sig")
desc

### Sınıf bazında ortalama ve medyan
Her sınıfın özellik değerleri ne kadar farklıysa model sınıfları o kadar kolay ayırır.

In [ ]:
by_class_mean = df.groupby("class_name")[FEATS].mean().reindex(CLASS_ORDER).round(4)
by_class_median = df.groupby("class_name")[FEATS].median().reindex(CLASS_ORDER).round(4)
by_class_mean.to_csv(f"{T}/sinif_bazinda_ortalama.csv", encoding="utf-8-sig")
by_class_median.to_csv(f"{T}/sinif_bazinda_medyan.csv", encoding="utf-8-sig")
display(by_class_mean)
display(by_class_median)

### Şehir bazında ortalama
Aynı sınıf farklı şehirlerde farklı görünebilir (ör. Artvin'in nemli ormanı ile Şanlıurfa'daki ağaçlık alan). Bu fark Faz 6'daki genelleme sorusunun temelidir.

In [ ]:
by_city_class = df.groupby(["class_name", "city"])[["NDVI", "NDBI", "MNDWI", "NDMI"]].mean().round(3)
by_city_class.to_csv(f"{T}/sehir_sinif_bazinda_indeks_ortalama.csv", encoding="utf-8-sig")
by_city_class.unstack("city")["NDVI"].reindex(CLASS_ORDER)

## 4. Dağılım grafikleri
### Histogramlar
Her özelliğin değerlerinin nasıl dağıldığını gösterir.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, f in zip(axes.ravel(), FEATS):
    sns.histplot(df[f], bins=50, ax=ax, color="steelblue")
    ax.axvline(df[f].mean(), color="red", ls="--", lw=1, label="ortalama")
    ax.axvline(df[f].median(), color="black", ls=":", lw=1, label="medyan")
    ax.set_title(f); ax.set_xlabel(""); ax.set_ylabel("")
axes[0, 0].legend(fontsize=8)
savefig("histogramlar")

**Neye bakmalıyız:** Çift tepeli (iki zirveli) dağılımlar var mı? Örneğin NDVI'de bir tepe bitkisiz alanlar, bir tepe bitkili alanlar olabilir. Bu, özelliğin sınıfları ayırdığını gösterir. Ortalama (kırmızı) ile medyan (siyah) arasındaki fark büyükse dağılım çarpıktır.

### Sınıflara göre kutu grafikleri (NDVI, NDBI, MNDWI)
**Kutu grafiği:** Kutu Q1–Q3 arasını (verinin ortadaki %50'si), kutudaki çizgi medyanı, bıyıklar 1.5 × IQR'ye kadar olan değerleri, noktalar ise aykırı değerleri gösterir.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, f in zip(axes, ["NDVI", "NDBI", "MNDWI"]):
    sns.boxplot(data=df, x="class_name", y=f, hue="class_name", legend=False, order=CLASS_ORDER,
                ax=ax, palette="tab10", fliersize=1)
    ax.set_xlabel(""); ax.set_title(f); ax.tick_params(axis="x", rotation=30)
savefig("kutu_grafikleri_indeksler")

**Neye bakmalıyız:**
- NDVI'de Orman ve Tarım'ın kutuları en yukarıda mı?
- MNDWI'de Su/Sulak alan sınıfı diğerlerinden açıkça ayrılıyor mu (çoğu değer 0'ın üstünde mi)?
- NDBI'de Yapılaşma ile Çıplak alan birbirine yakın mı? Literatürde yapı ile çıplak toprağın spektral olarak karıştığı sıkça belirtilir. Kutular üst üste biniyorsa modelin bu iki sınıfı karıştırmasını bekleriz (Faz 3 karışıklık matrisinde kontrol edin).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, f in zip(axes, ["NDVI", "NDMI"]):
    sns.boxplot(data=df, x="class_name", y=f, hue="city", order=CLASS_ORDER, hue_order=CITY_ORDER,
                ax=ax, fliersize=0.5, linewidth=0.6)
    ax.set_xlabel(""); ax.set_title(f"{f}: sınıf × şehir"); ax.tick_params(axis="x", rotation=30)
if axes[0].get_legend(): axes[0].get_legend().remove()
axes[1].legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
savefig("kutu_grafikleri_sehir")

**Neye bakmalıyız:** Aynı sınıfın kutuları şehirden şehre ne kadar kayıyor? Örneğin Tarım sınıfının NDVI'si sulu tarım yapılan illerde yüksek, kuru tarım yapılan illerde düşük olabilir. Kayma büyükse, bir şehirde eğitilen modelin başka bir şehirde zorlanmasını bekleriz (AS5, Faz 6).

## 5. Korelasyon ısı haritası
**Pearson korelasyonu** iki özelliğin birlikte artıp azalma eğilimini −1 ile +1 arasında ölçer. +1 tam aynı yönde, −1 tam ters yönde, 0 ise doğrusal ilişki yok demektir.

Çok yüksek korelasyonlu (|r| > 0.9) özellik çiftleri aynı bilgiyi taşır. Örneğin NDVI ile SAVI, ikisi de aynı iki banttan hesaplandığı için çok benzer.

In [ ]:
corr = df[FEATS].corr(method="pearson")
corr.round(3).to_csv(f"{T}/korelasyon_matrisi.csv", encoding="utf-8-sig")
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, square=True,
            annot_kws={"size": 7}, cbar_kws={"label": "Pearson r"})
plt.title("Özellikler arası korelasyon")
savefig("korelasyon_isi_haritasi")

pairs = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
             .rename("r").reset_index().rename(columns={"level_0": "ozellik_1", "level_1": "ozellik_2"}))
high = pairs[pairs["r"].abs() > 0.9].sort_values("r", key=abs, ascending=False)
high.to_csv(f"{T}/yuksek_korelasyonlu_ciftler.csv", index=False, encoding="utf-8-sig")
high

**Neye bakmalıyız:** Hangi çiftler |r| > 0.9? Görünür bantlar (B2, B3, B4) birbirleriyle, NDVI–SAVI çifti de güçlü korelasyon gösterebilir.

**NDBI ve NDMI özel bir durum: r = −1.** Landsat'ta iki indeks de aynı iki bantla (NIR = B5, SWIR1 = B6) hesaplanıyor:
- NDBI = (SWIR1 − NIR) / (SWIR1 + NIR)
- NDMI = (NIR − SWIR1) / (NIR + SWIR1)

Pay aynı farkın ters işaretlisi, payda aynı. Yani her pikselde **NDMI = −NDBI** ve ikisi tamamen aynı bilgiyi taşıyor (aşağıdaki hücre bunu sayıyla doğruluyor). Bu yüzden model girdisinden **NDMI çıkarıldı, NDBI tutuldu.** NDBI, Yapılaşma sınıfını doğrudan temsil ediyor ve arazi örtüsü literatüründe bu adla kullanılıyor. NDMI ise Faz 4'teki 1 km nem analizinde "nem göstergesi" olarak kalıyor. Orada aynı sayıya hangi adla baktığımızın yorum açısından önemi var.

**Diğer yüksek korelasyonlu çiftler için karar:** Bunları **çıkarmıyoruz.** Ağaç tabanlı modeller (RF, XGBoost) korelasyonlu özelliklerden etkilenmez. SVM ve MLP'de ise ölçekleme yapılıyor ve özellik sayısı (12) zaten az. Tek yan etkisi, özellik önemi grafiklerinde önemin korelasyonlu özellikler arasında **bölünmesidir**. Bunu Faz 3'te yorumlarken dikkate alacağız.

In [ ]:
diff = (df["NDMI"] + df["NDBI"]).abs()
print(f"max |NDMI + NDBI| = {diff.max():.2e}  (0'a eşitse NDMI = −NDBI tam olarak geçerli)")
print(f"Pearson r(NDMI, NDBI) = {df['NDMI'].corr(df['NDBI']):.6f}")

## 5b. Su/Sulak alan sınıfının iç yapısı: kalıcı su (80) ve otsu sulak alan (90)
"Su/Sulak alan" sınıfımız iki WorldCover sınıfının birleşimi. **Kalıcı su (80):** göl, baraj, deniz. **Otsu sulak alan (90):** sazlık, bataklık, su basan çayır. İkincisi yaz kompozitinde çoğu zaman **sudan çok bitki gibi görünür**: NDVI yüksek, MNDWI negatif. Çünkü yaz aylarında suyun üstü bitkiyle kaplıdır ya da su çekilmiştir.

Bu durum, sınıfın içinde spektral olarak birbirine benzemeyen iki grup olduğu anlamına gelir. Modelin otsu sulak alan noktalarını Çayır/Çalı veya Tarım ile karıştırmasını bekleriz. Faz 3'te bunu alt tip bazında ayrıca ölçüyoruz.

In [ ]:
wet = df[df["class_name"] == config.CLASSES[5]].copy()
wet["alt_tip"] = wet["wc_code"].map(config.WORLDCOVER_NAMES)
sub_counts = pd.crosstab(wet["city"], wet["alt_tip"]).reindex(CITY_ORDER).fillna(0).astype(int)
sub_counts["otsu_sulak_yuzde"] = (100 * sub_counts.get(config.WORLDCOVER_NAMES[90], 0)
                                  / sub_counts.sum(axis=1)).round(1)
sub_stats = wet.groupby("alt_tip")[["NDVI", "MNDWI", "NDBI", "SR_B5", "SR_B6"]].agg(["mean", "median"]).round(3)
sub_counts.to_csv(f"{T}/su_sulak_alt_tip_sayilari.csv", encoding="utf-8-sig")
sub_stats.to_csv(f"{T}/su_sulak_alt_tip_istatistik.csv", encoding="utf-8-sig")
display(sub_counts)
display(sub_stats)

# Karşılaştırma: alt tipler + Çayır/Çalı + Tarım
ref = df[df["class_name"].isin([config.CLASSES[3], config.CLASSES[1]])].assign(alt_tip=lambda d: d["class_name"])
comp = pd.concat([wet, ref])
order = [config.WORLDCOVER_NAMES[80], config.WORLDCOVER_NAMES[90], config.CLASSES[3], config.CLASSES[1]]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, f in zip(axes, ["NDVI", "MNDWI"]):
    sns.boxplot(data=comp, x="alt_tip", y=f, order=order, hue="alt_tip", legend=False,
                palette=["#1f77b4", "#17becf", "#bcbd22", "#2ca02c"], ax=ax, fliersize=1)
    ax.axhline(0, color="grey", lw=0.8, ls="--"); ax.set_xlabel(""); ax.set_title(f)
savefig("su_sulak_alt_tipler")

**Neye bakmalıyız:**
- Otsu sulak alanın (90) NDVI kutusu Kalıcı su'dan (80) çok Çayır/Çalı ve Tarım'a mı yakın? MNDWI kutusu 0'ın altında mı?
- Hangi illerde otsu sulak alan payı yüksek? Konya ve Van'daki sulak alanlar AS3 için önemli. Bu payın yüksek olduğu illerde Su/Sulak sınıfının başarısı düşük çıkabilir.

Rapor için not `report/sinirliliklar_notlari.md` içinde.

## 6. Şehirlerin gerçek renkli Landsat görüntüleri (2014 ve 2024)

In [ ]:
import matplotlib.image as mpimg
years = config.CHANGE_YEARS
fig, axes = plt.subplots(len(years), len(CITY_ORDER), figsize=(3.2 * len(CITY_ORDER), 3.4 * len(years)))
for j, city in enumerate(CITY_ORDER):
    for i, y in enumerate(years):
        ax = axes[i, j]
        p = f"{F}/rgb/{city}_{y}.png"
        if os.path.exists(p):
            ax.imshow(mpimg.imread(p))
        ax.set_title(f"{city} {y}", fontsize=10); ax.axis("off")
plt.suptitle("Landsat yaz medyan kompozitleri (gerçek renk: B4-B3-B2)", y=1.01)
savefig("rgb_kompozitler")

**Neye bakmalıyız:** Kompozitler bulutsuz ve dikişsiz mi? Bulut kalıntısı (beyaz lekeler) ya da şerit izleri varsa not edin. Çıplak gözle görülebilen değişimler var mı? Örneğin baraj gölü seviyesi, şehir yayılımı ya da Konya'daki göllerin küçülmesi. Bunları Faz 4'ün sayısal sonuçlarıyla karşılaştırın.

## 7. Veri kalitesi kontrolleri
### 7a. Eksik değer
Örnekler Earth Engine'de alınırken bulutlu/eksik pikseller zaten atıldı. Burada eksik değer çıkmaması gerekir.

In [ ]:
missing = df[FEATS + ["class", "city", "lon", "lat"]].isna().sum().rename("eksik_sayi").to_frame()
missing["eksik_yuzde"] = 100 * missing["eksik_sayi"] / len(df)
missing.to_csv(f"{T}/eksik_deger.csv", encoding="utf-8-sig")
print("Toplam eksik hücre:", int(missing["eksik_sayi"].sum()))
missing.T

### 7b. Aykırı değer (IQR yöntemi)
Bir değer `Q1 − 1.5×IQR` altında ya da `Q3 + 1.5×IQR` üstündeyse aykırı sayılır. IQR = Q3 − Q1. Aykırı değerleri hem **tüm veride** hem de **sınıf içinde** sayıyoruz. Tüm veride "aykırı" görünen bir değer, çoğu zaman sadece nadir bir sınıfa aittir. Örneğin su piksellerinin MNDWI'si tüm veriye göre aykırı görünebilir ama su sınıfı için normaldir.

In [ ]:
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr))

out_all = pd.Series({f: iqr_outliers(df[f]).mean() * 100 for f in FEATS}, name="tum_veri_yuzde")
out_cls = pd.Series({f: df.groupby("class_name")[f].transform(iqr_outliers).mean() * 100 for f in FEATS},
                    name="sinif_ici_yuzde")
# Fiziksel olarak imkânsız değerler: yansıma 0-1 dışında, indeks -1..1 dışında (SAVI -1.5..1.5)
phys = pd.Series({f: (((df[f] < 0) | (df[f] > 1)) if f.startswith("SR_")
                      else (df[f].abs() > (1.5 if f == "SAVI" else 1))).mean() * 100 for f in FEATS},
                 name="fiziksel_aralik_disi_yuzde")
outliers = pd.concat([out_all, out_cls, phys], axis=1).round(2)
outliers.to_csv(f"{T}/aykiri_deger.csv", encoding="utf-8-sig")
outliers

### 7c. Sınıf dengesizliği
**Dengesizlik oranı** = en kalabalık sınıfın örnek sayısı / en az örnekli sınıfın örnek sayısı. 1'e yakınsa veri dengelidir. Kabaca 3'ün üstü belirgin dengesizlik sayılır.

In [ ]:
cnt = pd.crosstab(df["city"], df["class_name"]).reindex(index=CITY_ORDER, columns=CLASS_ORDER).fillna(0)
imb = pd.DataFrame({"en_az": cnt.min(axis=1), "en_cok": cnt.max(axis=1)})
imb["dengesizlik_orani"] = (imb["en_cok"] / imb["en_az"].replace(0, np.nan)).round(2)
tot = df["class_name"].value_counts()
imb.loc["TUMU"] = [tot.min(), tot.max(), round(tot.max() / tot.min(), 2)]
imb.to_csv(f"{T}/sinif_dengesizligi.csv", encoding="utf-8-sig")
imb

## 8. Ön işleme kararları (gerekçeleriyle)
Aşağıdaki tablo Faz 3'te uygulanacak kararları listeliyor. Kontrol sonuçlarına (7a–7c) göre tablo kendini günceller.

In [ ]:
n_missing = int(missing["eksik_sayi"].sum())
max_imb = imb.loc["TUMU", "dengesizlik_orani"]
decisions = pd.DataFrame([
    ("Eksik değer",
     "Yok — işlem yapılmadı" if n_missing == 0 else f"{n_missing} eksik hücre → ilgili satırlar çıkarılacak",
     "Bulutlu/eksik pikseller örnekleme sırasında zaten atıldı"),
    ("Aykırı değer", "Çıkarılmadı",
     "Aykırı görünen değerlerin çoğu nadir sınıflara (su, çıplak) ait ve gerçek yüzey özelliğidir; "
     "çıkarmak o sınıfları zayıflatır. RF/XGBoost aykırı değere dayanıklıdır; SVM/MLP için ölçekleme yapılır"),
    ("Ölçekleme", "StandardScaler (ortalama 0, std 1) — yalnızca SVM ve MLP",
     "Bu iki model mesafe/gradyan tabanlıdır ve özelliklerin aynı ölçekte olmasını ister. "
     "Ölçekleyici sadece eğitim verisiyle fit edilir (Pipeline içinde) → veri sızıntısı yok"),
    ("Sınıf dengesizliği",
     f"Dengesizlik oranı (tümü) = {max_imb}; ek dengeleme yapılmadı" if max_imb < 3
     else f"Dengesizlik oranı (tümü) = {max_imb}; class_weight='balanced' kullanılacak",
     "Örnekleme zaten sınıf dengeli yapıldı; metriklerde macro ortalama (her sınıfa eşit ağırlık) da raporlanır"),
    ("NDMI", "Model girdisinden çıkarıldı (NDBI tutuldu)",
     "Landsat'ta NDMI = −NDBI (aynı NIR ve SWIR1 bantları) → aynı bilgi; NDBI yapılaşma sınıfını doğrudan temsil ediyor. "
     "NDMI Faz 4 nem analizinde kullanılmaya devam ediyor"),
    ("Diğer yüksek korelasyonlu özellikler", f"{len(high)} çift |r|>0.9 (NDBI–NDMI dahil) — diğerleri çıkarılmadı",
     "Ağaç modelleri etkilenmez; özellik sayısı az; önem grafikleri yorumlanırken dikkate alınacak"),
    ("Veri bölme", "%70 eğitim / %15 doğrulama / %15 test, şehir+sınıf'a göre katmanlı",
     "Her kümede her şehir ve sınıf aynı oranda temsil edilir"),
], columns=["Konu", "Karar", "Gerekçe"])
decisions.to_csv(f"{T}/on_isleme_kararlari.csv", index=False, encoding="utf-8-sig")
pd.set_option("display.max_colwidth", None)
decisions

## 9. İklim ve grid verisinin kısa tanıtımı
Faz 4'te kullanılacak iki veri setinin özet istatistikleri.

In [ ]:
climate = pd.read_csv(f"{config.DATA_PROCESSED}/iklim_sehir_yillik.csv")
clim_cols = [c for c in climate.columns if c not in ("city", "year")]
clim_desc = describe_full(climate[clim_cols]).round(3)
clim_desc.to_csv(f"{T}/iklim_tanimlayici_istatistik.csv", encoding="utf-8-sig")
print("İklim serisi:", climate.shape, "| yıllar:", climate["year"].min(), "-", climate["year"].max())
display(clim_desc)

grid = pd.read_csv(f"{config.DATA_PROCESSED}/grid_1km_2014_2024.csv")
grid_cols = [f"{b}_{y}" for y in config.CHANGE_YEARS for b in ["NDVI", "NDMI", "LST", "modis_lst_day"]] + ["elevation"]
grid_desc = describe_full(grid[grid_cols]).round(3)
grid_desc.to_csv(f"{T}/grid_tanimlayici_istatistik.csv", encoding="utf-8-sig")
print("1 km grid:", grid.shape)
display(grid.groupby("city").size().rename("hucre_sayisi").to_frame().T)
grid_desc

## 10. Üretilen dosyalar

In [ ]:
for p in sorted(glob.glob(f"{T}/*.csv")):
    print(p)
for p in sorted(glob.glob(f"{F}/*.png")):
    print(p)

## Faz 2 bitti mi?
- [ ] Veri seti özeti, sınıf dağılımı ve tanımlayıcı istatistik tabloları `results/tables/` altında
- [ ] Sınıf dağılımı, histogram, kutu grafikleri, korelasyon ısı haritası, RGB kompozitler `results/figures/` altında
- [ ] Her grafiğin altına kendi gözleminizi not ettiniz (rapora girecek)

## Bu fazda ne yaptık, neden yaptık
- Modele girmeden önce veriyi tanıdık: kaç örnek, kaç özellik, sınıflar nasıl dağılmış. Hocanın istediği ortalama/medyan/mod tablosu bu fazdan çıkar.
- Sınıfların indeks değerlerini kutu grafikleriyle karşılaştırdık. Hangi sınıfların kolay, hangilerinin zor ayrılacağını (ör. yapı ile çıplak toprak) model kurmadan önce tahmin edebiliriz.
- Korelasyon analiziyle aynı bilgiyi taşıyan özellikleri belirledik ve neden çıkarmadığımızı gerekçelendirdik.
- Eksik değer, aykırı değer ve dengesizliği kontrol ettik. Ön işleme kararlarını (ölçekleme yalnızca eğitim verisiyle, aykırı değerleri tutma) yazılı hale getirdik.
- Aynı sınıfın şehirden şehre farklı göründüğünü gösterdik. Bu, Faz 6'daki genelleme sorusunun (AS5) gerekçesi.